In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import remez, freqz, find_peaks
from ipywidgets import Dropdown, FloatSlider, HBox, VBox, HTML, Layout, interactive_output
from IPython.display import display, HTML as DisplayHTML

# ============================================================
# REMEZ ALGORITHM — TYPE-IV FIR EQUIRIPPLE APPROXIMATION
# ============================================================

plt.rcParams.update({'font.size':11.5,'axes.titlesize':13.5,'axes.labelsize':11.5,'xtick.labelsize':10.5,'ytick.labelsize':10.5,'legend.fontsize':9.4})

# ============================================================
# REMOVE HORIZONTAL SCROLL BARS
# ============================================================

display(DisplayHTML("""
<style>

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child {
    overflow-x: visible !important;
    overflow-y: visible !important;
    max-width: none !important;
}

.jp-OutputArea,
.output_wrapper,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
}

.widget-box,
.jupyter-widgets,
.jupyter-widget,
.widget-subarea {
    overflow: visible !important;
    max-width: none !important;
}

.jp-OutputArea-child {
    width: auto !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<style>

.rm-root{
    width:1120px;
    max-width:1120px;
    font-family:Arial,sans-serif;
}

.rm-header{
    background:linear-gradient(90deg,#7b1fa2,#9c27b0);
    color:white;
    padding:10px 15px;
    border-radius:8px 8px 0 0;
    font-size:18px;
    font-weight:bold;
}

.rm-doc{
    background:#fbf7fc;
    border:1px solid #d7c4e2;
    border-top:none;
    padding:10px 13px;
    border-radius:0 0 8px 8px;
    font-size:13.5px;
    line-height:1.55;
    margin-bottom:9px;
}

.rm-title{
    font-weight:bold;
    color:#6a1b9a;
    font-size:14px;
    margin-bottom:6px;
}

.rm-info{
    width:1094px;
    border:1px solid #d7c4e2;
    border-radius:7px;
    padding:10px 12px;
    font-size:13.5px;
    line-height:1.5;
}

.rm-box{
    flex:1;
    min-width:245px;
}

</style>

<div class="rm-root">

<div class="rm-header">
Remez Algorithm for FIR - Type IV — Equiripple Approximation and Alternation Theorem
</div>

<div class="rm-doc">

<b>Purpose.</b>
This notebook demonstrates equiripple FIR-Type IV filter design using the Remez exchange algorithm.
A Type-IV linear-phase FIR filter has an <b>even length</b> and an <b>antisymmetric impulse response</b>.

<br><br>

<b>Structural property.</b>
For a Type-IV FIR filter,
<b>h[n] = -h[N-1-n]</b>.
Consequently, the frequency response necessarily satisfies
<b>H(e<sup>j0</sup>) = 0</b>.
Unlike a Type-III filter, however, the response at ω = π is not forced to zero.

<br><br>

<b>Filter used in this demonstration.</b>
Because a Type-IV filter has a forced zero at ω = 0 but no forced zero at ω = π,
a high-pass specification provides a natural demonstration of this structure.

<br><br>

<b>What the controls show.</b>
The parameter <b>N</b> is the even FIR length.
The parameters <b>ωs</b> and <b>ωp</b> define the stopband and passband edges.
The ratio <b>Ws/Wp</b> controls the relative importance of the stopband error,
with <b>Wp = 1</b> fixed.

<br><br>

<b>What to observe.</b>
The weighted approximation error exhibits the equiripple property in the active frequency bands.
The impulse response is antisymmetric about a point located halfway between the two central samples.
The response is forced to zero at ω = 0, but not at ω = π.

</div>

</div>
""")

# ============================================================
# CONTROLS
# ============================================================

N_control = Dropdown(options=[10,12,14,16,18,20,22,24,26,28,30,32,36,42,52],value=20,description='N:',style={'description_width':'25px'},layout=Layout(width='150px'))

ws_control = FloatSlider(value=0.40,min=0.10,max=0.75,step=0.01,description='ωs / π:',continuous_update=True,readout_format='.2f',style={'description_width':'55px'},layout=Layout(width='260px'))

wp_control = FloatSlider(value=0.50,min=0.15,max=0.90,step=0.01,description='ωp / π:',continuous_update=True,readout_format='.2f',style={'description_width':'55px'},layout=Layout(width='260px'))

weight_control = FloatSlider(value=2.0,min=0.25,max=8.0,step=0.25,description='Ws / Wp:',continuous_update=True,readout_format='.2f',style={'description_width':'65px'},layout=Layout(width='280px'))

controls = VBox([
    HTML('<div class="rm-title">Interactive controls</div>'),
    HBox([N_control,ws_control,wp_control,weight_control],layout=Layout(width='1088px',justify_content='space-between',align_items='center',overflow='visible'))
],layout=Layout(width='1120px',border='1px solid #d7c4e2',padding='9px 12px',margin='0 0 8px 0',overflow='visible'))

info = HTML(layout=Layout(width='1120px',margin='0 0 8px 0',overflow='visible'))

# ============================================================
# DESIGN FUNCTIONS
# ============================================================

def calculate_remez_design(N,ws,wp,Ws):
    h = remez(N,[0.0,ws,wp,1.0],[0.0,1.0],weight=[Ws,1.0],type='hilbert',fs=2.0,maxiter=100,grid_density=32)

    omega,H = freqz(h,worN=8192)
    f = omega/np.pi

    A = np.imag(H*np.exp(1j*omega*(N-1)/2))

    active = (f <= ws) | (f >= wp)

    Hd = np.full_like(f,np.nan)
    W = np.full_like(f,np.nan)
    E = np.full_like(f,np.nan)

    Hd[f <= ws] = 0.0
    Hd[f >= wp] = 1.0

    W[f <= ws] = Ws
    W[f >= wp] = 1.0

    E[active] = W[active]*(Hd[active]-A[active])

    return h,omega,f,A,Hd,W,E

# ============================================================
# EXTREMAL FREQUENCIES
# ============================================================

def find_error_extrema(f,E,ws,wp):
    extrema = []

    masks = [
        f <= ws,
        f >= wp
    ]

    for mask in masks:
        idx = np.where(mask)[0]

        if len(idx) == 0:
            continue

        y = E[idx]

        pmax,_ = find_peaks(y)
        pmin,_ = find_peaks(-y)

        local = np.unique(np.concatenate(([0],pmax,pmin,[len(idx)-1])))
        extrema.extend(idx[local])

    return np.array(sorted(set(extrema)))

# ============================================================
# MAIN INTERACTIVE FUNCTION
# ============================================================

def draw_remez(N,ws,wp,Ws):
    if wp <= ws:
        return

    h,omega,f,A,Hd,W,E = calculate_remez_design(N,ws,wp,Ws)
    extrema = find_error_extrema(f,E,ws,wp)

    M = N//2
    L = M-1
    required_extrema = M+1

    stop_mask = f <= ws
    pass_mask = f >= wp

    ds = np.max(np.abs(A[stop_mask]))
    dp = np.max(np.abs(A[pass_mask]-1.0))
    Emax = np.nanmax(np.abs(E))
    ratio = Ws*ds/dp if dp > 1e-14 else np.nan

    antisymmetry_error = np.max(np.abs(h+h[::-1]))

    H_zero = np.sum(h)
    H_pi = np.sum(h*np.exp(-1j*np.pi*np.arange(N)))

    center = (N-1)/2

    info.value = f"""
    <div class="rm-info">

    <div class="rm-title">
    Current equiripple design
    </div>

    <div style="display:flex;flex-wrap:nowrap;justify-content:space-between;gap:18px;">

        <div class="rm-box">
            FIR length: <b>N = {N}</b><br>
            Type-IV parameter: <b>M = {M}</b><br>
            Polynomial degree: <b>L = {L}</b><br>
            Alternation requirement:
            <b style="white-space:nowrap;">at least M+1 = {required_extrema} extrema</b>
        </div>

        <div class="rm-box">
            Stopband edge: <b>ωs = {ws:.2f}π</b><br>
            Passband edge: <b>ωp = {wp:.2f}π</b><br>
            Transition width: <b>{wp-ws:.2f}π</b>
        </div>

        <div class="rm-box">
            Wp = <b>1</b><br>
            Ws = <b>{Ws:.2f}</b><br>
            Maximum weighted error: <b>{Emax:.6f}</b>
        </div>

        <div class="rm-box">
            Passband deviation δp: <b>{dp:.6f}</b><br>
            Stopband deviation δs: <b>{ds:.6f}</b><br>
            Ws δs / δp: <b>{ratio:.4f}</b>
        </div>

    </div>

    <div style="margin-top:8px;padding-top:7px;border-top:1px solid #e2d4e8;">
        <b>Structural constraints:</b>
        h[n] = -h[N-1-n]
        &nbsp;&nbsp; | &nbsp;&nbsp;
        antisymmetry center = {center:.1f}
        &nbsp;&nbsp; | &nbsp;&nbsp;
        antisymmetry error = {antisymmetry_error:.2e}
        <br>
        <b>Endpoint behavior:</b>
        |H(e<sup>j0</sup>)| = {abs(H_zero):.2e}
        &nbsp;&nbsp; | &nbsp;&nbsp;
        |H(e<sup>jπ</sup>)| = {abs(H_pi):.6f}
    </div>

    </div>
    """

    # ========================================================
    # FIGURE
    # ========================================================

    fig,axes = plt.subplots(2,2,figsize=(11.6,7.9))
    ax1,ax2,ax3,ax4 = axes.flat

    # ========================================================
    # 1. OPTIMAL MAGNITUDE RESPONSE
    # ========================================================

    y_top = max(1.12,1.08*np.max(np.abs(A)),1+2.5*dp)

    ax1.plot(f,np.abs(A),color='red',linewidth=1.6,label='Optimal FIR response')

    ax1.plot([0,ws],[0,0],'--',linewidth=1.0,label='Desired response')
    ax1.plot([wp,1],[1,1],'--',linewidth=1.0)

    ax1.axvspan(ws,wp,alpha=0.07,label="Don't-care region")

    ax1.axhline(1+dp,linestyle=':',linewidth=0.9)
    ax1.axhline(1-dp,linestyle=':',linewidth=0.9)
    ax1.axhline(ds,linestyle=':',linewidth=0.9)

    ax1.plot(0.0,0.0,'ko',markersize=5,label=r'Forced zero at $\omega=0$')

    ax1.set_xlim(0,1)
    ax1.set_ylim(-0.05,y_top)

    ax1.set_title('Optimal Equiripple Magnitude Response')
    ax1.set_xlabel(r'Normalized frequency $\omega/\pi$')
    ax1.set_ylabel(r'$|H(e^{j\omega})|$')

    ax1.grid(True,linestyle=':',alpha=0.25)

    ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

    # ========================================================
    # 2. WEIGHT FUNCTION
    # ========================================================

    ax2.plot([0,ws],[Ws,Ws],color='red',linewidth=2)
    ax2.plot([wp,1],[1,1],color='red',linewidth=2)

    ax2.axvspan(ws,wp,alpha=0.07)

    ax2.set_xlim(0,1)
    ax2.set_ylim(0,max(1.15,1.15*Ws))

    ax2.set_title('Weight Function')
    ax2.set_xlabel(r'Normalized frequency $\omega/\pi$')
    ax2.set_ylabel(r'$W(\omega)$')

    ax2.grid(True,linestyle=':',alpha=0.25)

    # ========================================================
    # 3. WEIGHTED ERROR
    # ========================================================

    ax3.plot(f,E,color='red',linewidth=1.5,label='Weighted error')
    ax3.plot(f[extrema],E[extrema],'o',markersize=4,label='Extremal frequencies')

    ax3.axhline(Emax,linestyle='--',linewidth=1.0,label=r'$\pm E_{max}$')
    ax3.axhline(-Emax,linestyle='--',linewidth=1.0)

    ax3.axvspan(ws,wp,alpha=0.07)

    ax3.set_xlim(0,1)
    ax3.set_ylim(-1.25*Emax,1.25*Emax)

    ax3.set_title('Weighted Error and Alternation')
    ax3.set_xlabel(r'Normalized frequency $\omega/\pi$')
    ax3.set_ylabel(r'$E(\omega)$')

    ax3.grid(True,linestyle=':',alpha=0.25)

    ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.20),ncol=2,frameon=False)

    # ========================================================
    # 4. IMPULSE RESPONSE
    # ========================================================

    n = np.arange(N)

    markerline,stemlines,baseline = ax4.stem(n,h,linefmt='r-',markerfmt='ro',basefmt=' ')

    plt.setp(markerline,markersize=4)
    plt.setp(stemlines,linewidth=1.0)

    ax4.axvline(center,linestyle='--',linewidth=1.0,label=f'Antisymmetry center = {center:.1f}')

    ax4.axhline(0,linewidth=0.8)

    ax4.set_title('Type-IV FIR Impulse Response')
    ax4.set_xlabel('Sample index $n$')
    ax4.set_ylabel('$h[n]$')

    ax4.grid(True,linestyle=':',alpha=0.25)

    ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.20),frameon=False)

    # ========================================================
    # LAYOUT
    # ========================================================

    plt.subplots_adjust(left=0.07,right=0.98,top=0.94,bottom=0.10,wspace=0.28,hspace=0.64)

    plt.show()
    plt.close(fig)

# ============================================================
# SLIDER CONSISTENCY
# ============================================================

def update_ws(change):
    wp_control.min = min(0.90,ws_control.value+0.05)

    if wp_control.value <= ws_control.value:
        wp_control.value = min(0.90,ws_control.value+0.10)

def update_wp(change):
    ws_control.max = max(0.10,wp_control.value-0.05)

    if ws_control.value >= wp_control.value:
        ws_control.value = max(0.10,wp_control.value-0.10)

ws_control.observe(update_ws,names='value')
wp_control.observe(update_wp,names='value')

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

plots = interactive_output(draw_remez,{'N':N_control,'ws':ws_control,'wp':wp_control,'Ws':weight_control})

plots.layout = Layout(width='auto',max_width='none',overflow='visible')

# ============================================================
# DISPLAY
# ============================================================

display(documentation)
display(controls)
display(info)
display(plots)